# 블로그 및 정리한 내용 복습
이전에 정리한 개념들을 코드로 치며 복습합니다.


## RNN의 전파 및 기울기 소실/폭발

[링크](https://app.notion.com/p/3eb117a41d4d80b69a7fcf9bf1f52ce9)

In [1]:
import token

import torch
from torch import optim

torch.manual_seed(0)

# 들어갈 input의 벡터 크기
input_size= 2
# RNN 내부에서 변형되는 상태로 그림상 rollover 했을 때 가로로 이동하는 상태
hidden_size = 3
# 문장에 또는 데이터의 길이
seq_len = 4

In [2]:
x = torch.tensor([
    [1.0,  0.5],
    [0.2, -0.3],
    [0.8,  0.1],
    [-0.4, 0.7]
])

In [3]:
# 입력 x를 hidden 차원으로 변경해주기 위한 행렬
W_xh = torch.randn(hidden_size, input_size)
# 기존의 hidden state를 새로 갱신하기 위한 행렬
W_hh = torch.randn(hidden_size, hidden_size)

b = torch.zeros(hidden_size)

h = torch.zeros(hidden_size)

W_xh.shape, W_hh.shape, b.shape, h.shape

(torch.Size([3, 2]), torch.Size([3, 3]), torch.Size([3]), torch.Size([3]))

In [4]:
# [hidden_size, input_size] @ [input_size, ] -> [hidden_size=3, ]
(W_xh @ x[0]).shape

torch.Size([3])

In [5]:
hidden_states = []

# sequence 내부의 데이터를 순회하면서 hidden_state를 갱신해나가주기
for t in range(seq_len):
    x_t = x[t]

    net = W_xh @ x_t + W_hh @ h + b

    h = torch.tanh(net)

    hidden_states.append(h)

    print(f"\nt = {t} | x_t = {x_t} | net = {net} | h = {h}")

# [seq, hidden_size]
torch.vstack(hidden_states).shape


t = 0 | x_t = tensor([1.0000, 0.5000]) | net = tensor([ 1.3943, -1.8946, -1.7838]) | h = tensor([ 0.8841, -0.9558, -0.9451])

t = 1 | x_t = tensor([ 0.2000, -0.3000]) | net = tensor([ 0.6316, -0.5647, -0.5943]) | h = tensor([ 0.5592, -0.5114, -0.5330])

t = 2 | x_t = tensor([0.8000, 0.1000]) | net = tensor([ 1.3837, -1.7036, -1.4785]) | h = tensor([ 0.8818, -0.9359, -0.9012])

t = 3 | x_t = tensor([-0.4000,  0.7000]) | net = tensor([-0.6022,  1.3081, -1.3653]) | h = tensor([-0.5386,  0.8638, -0.8776])


torch.Size([4, 3])

In [6]:
import torch.nn as nn

# 문자열을 받아서 긍정/부정으로 만들어주기

class MyRNN(nn.Module):
    def __init__(self, input_size=32, hidden_size=64, num_layers=1):
        super().__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=0.3 if num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        output, hidden = self.rnn(x.transpose(0, 1), h)
        # output.shape = [S, B, hidden_size]
        result = self.fc(output[-1, :, :])
        return result

In [7]:
# NSMT 입력 받기
import urllib

# urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings.txt", filename="ratings.txt")

In [8]:
import pandas as pd

df = pd.read_table("ratings.txt")#, names=["document", "labels"], encoding="utf_8")

df.head()

,id,document,label
0,8112052,어릴때보고 지금다시봐도 재밌어요ㅋㅋ,1
1,8132799,"디자인을 배우는 학생으로, 외국디자이너와 그들이 일군 전통을 통해 발전해가는 문화산...",1
2,4655635,폴리스스토리 시리즈는 1부터 뉴까지 버릴께 하나도 없음.. 최고.,1
3,9251303,와.. 연기가 진짜 개쩔구나.. 지루할거라고 생각했는데 몰입해서 봤다.. 그래 이런...,1
4,10067386,안개 자욱한 밤하늘에 떠 있는 초승달 같은 영화.,1


In [9]:
# 20만개
df.shape
df = df[:30_000]

In [10]:
sample = df["document"].iloc[0]
sample

'어릴때보고 지금다시봐도 재밌어요ㅋㅋ'

In [11]:
from mecab import MeCab

mecab = MeCab()
mecab.morphs(sample)

['어릴', '때', '보', '고', '지금', '다시', '봐도', '재밌', '어요', 'ㅋㅋ']

In [12]:
# 데이터 분할하기
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df[["document"]], df["label"],
    test_size=0.2,
    random_state=42
)
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((24000, 1), (24000,), (6000, 1), (6000,))

In [15]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42
)
X_train.shape, y_train.shape, X_val.shape, y_val.shape

((19200, 1), (19200,), (4800, 1), (4800,))

In [ ]:
vocab = dict()

for (idx, document) in enumerate(X_train["document"]):
    for w in mecab.morphs(document):
        if w not in vocab:
            vocab[w] = len(vocab)

    if idx % 500 == 0:
        print(f"{idx} | vocab: {len(vocab)}")

print(len(vocab))

In [17]:
vocab["<PAD>"] = len(vocab)
vocab["<UNK>"] = len(vocab)

In [41]:
# 임베딩 레이어 추가해주기

class MyRNN(nn.Module):
    def __init__(
            self,
            vocab_size=28743,
            embedding_dim=512,
            hidden_size=64,
            num_layers=1
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.input_size = embedding_dim
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(
            input_size=self.input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=0.3 if num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x, lengths):
        x = self.embedding(x)

        h = torch.zeros(
            self.num_layers, x.size(0), self.hidden_size,
            device=x.device
        )

        output, hidden = self.rnn(x.transpose(0, 1), h)
        # output.shape = [S, B, hidden_size]
        batch_idx = torch.arange(x.size(0))
        last_idx = lengths.to(x.device) - 1

        # [B, hidden_dim] -> [B, 1] -> [B, ]
        result = self.fc(output[last_idx, batch_idx, :]).squeeze(-1)
        return result

In [42]:
# vocab로 바꾸고 학습시키는 과정 만들기
max_seq = max(X_train["document"], key=lambda x: len(mecab.morphs(x)))

In [43]:
len(mecab.morphs(max_seq))

98

In [44]:
sum([len(mecab.morphs(l)) for l in X_train["document"]]) / len(X_train)

17.953333333333333

In [45]:
len(vocab)

19127

In [46]:
from torch.utils.data import Dataset

class NSMCDataset(Dataset):
    def __init__(self, x, y, vocab):
        if len(x) != len(y):
            raise ValueError(f"len(x) and len(y) must have same len | len(x): {len(x)}, len(y): {len(y)}")
        self.x = x
        self.y = y
        self.vocab = vocab

    def __getitem__(self, idx):
        label = self.y.iloc[idx]
        document = self.x["document"].iloc[idx]
        tokenized = [
            self.vocab.get(w, vocab.get("<UNK>"))
            for w
            in mecab.morphs(document)
        ]

        return tokenized, label

    def __len__(self):
        return len(self.y)


In [47]:
datasets = {
    "train": NSMCDataset(X_train, y_train, vocab),
    "val": NSMCDataset(X_val, y_val, vocab),
    "test": NSMCDataset(X_test, y_test, vocab),
}

In [51]:
from torch.utils.data import DataLoader
from functools import partial

# 길이가 다른 [(X, y), ...] 을 받아서 (x_batch, y_batch)로 만들어주는 함수
def collate_fn(batch, seq_length, pad_id):
    xs = []
    ys = []
    lengths = []

    for x, y in batch:
        length = min(len(x), seq_length)
        lengths.append(length)

        x = x[:seq_length]
        x = x + [pad_id] * (seq_length - len(x))

        xs.append(x)
        ys.append(y)

    return (
        torch.tensor(xs, dtype=torch.long),
        torch.tensor(ys, dtype=torch.float32),
        torch.tensor(lengths, dtype=torch.long)
    )

loaders = {
    "train": DataLoader(
        datasets["train"],
        batch_size=32,
        shuffle=True,
        collate_fn=partial(
            collate_fn,
            seq_length=100,
            pad_id=vocab["<PAD>"]
        )
    ),
    "val": DataLoader(
        datasets["val"],
        batch_size=32,
        collate_fn=partial(
            collate_fn,
            seq_length=100,
            pad_id=vocab["<PAD>"]
        )
    ),
    "test": DataLoader(
        datasets["test"],
        batch_size=32,
        collate_fn=partial(
            collate_fn,
            seq_length=100,
            pad_id=vocab["<PAD>"]
        )
    ),
}

In [52]:
from torch import optim

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = MyRNN(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_size=256
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.01,
    weight_decay=0.001,
    betas=(0.9, 0.999)
)

DEVICE

device(type='cuda')

In [53]:
EPOCH = 10

for epoch in range(1, EPOCH+1):
    # train
    model.train()
    for X, targets, lengths in loaders["train"]:
        X = X.to(DEVICE)
        targets = targets.to(DEVICE)

        logits = model(X, lengths) # [B, ]
        loss = criterion(logits, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # val
    total_loss = 0.0
    total_acc = 0.0
    total_batch = 0
    model.eval()
    with torch.inference_mode():
        for X, targets, lengths in loaders["val"]:
            X = X.to(DEVICE)
            targets = targets.to(DEVICE)
            logits = model(X, lengths)

            loss = criterion(logits, targets)

            batch_size = len(targets)
            total_batch += batch_size
            total_loss += loss * batch_size
            total_acc += (
                (logits > 0.0).long()
                ==
                targets.long()
            ).sum()

    if epoch % 1 == 0:
        print(f"epoch: {epoch:2d} "
              f"| loss: {total_loss / total_batch:.4f}"
              f"| acc : {total_acc / total_batch:.4f}")

epoch:  1 | loss: 0.0009| acc : 0.9998
epoch:  2 | loss: 0.0009| acc : 0.9998
epoch:  3 | loss: 0.0009| acc : 0.9998
epoch:  4 | loss: 0.0009| acc : 0.9998
epoch:  5 | loss: 0.0009| acc : 0.9998
epoch:  6 | loss: 0.0009| acc : 0.9998
epoch:  7 | loss: 0.0009| acc : 0.9998
epoch:  8 | loss: 0.0009| acc : 0.9998
epoch:  9 | loss: 0.0009| acc : 0.9998
epoch: 10 | loss: 0.0009| acc : 0.9998


In [87]:
it = iter(loaders["val"])
for i in range(3):
    next(it)
X, y, lengths = next(it)
X, y

(tensor([[ 1104,    34,   252,  ..., 19125, 19125, 19125],
         [  122,  1258,  1206,  ..., 19125, 19125, 19125],
         [   34,   697,   257,  ..., 19125, 19125, 19125],
         ...,
         [    1,    21,  1507,  ..., 19125, 19125, 19125],
         [  842,   660,   146,  ..., 19125, 19125, 19125],
         [ 1782,    65,    47,  ..., 19125, 19125, 19125]]),
 tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]))

In [85]:
id_to_token = {
    idx: token
    for token, idx
    in vocab.items()
}
lengths[0], [id_to_token[idx] for idx in X[0].tolist() if id_to_token[idx] != "<PAD>"]

(tensor(3), ['잼', '났', '어요'])

In [62]:
y[0]

tensor(1.)

In [68]:
type(lengths)

torch.Tensor

In [67]:
with torch.inference_mode():
    result = model(
        X[[0]].to(DEVICE),
        torch.tensor(19)
    )
result

tensor([16.5033], device='cuda:0')

In [88]:
import pandas as pd
with torch.inference_mode():
    result = model(X.to(DEVICE), lengths)

test_documents = []
for x in X:
    x = [id_to_token[idx.item()] for idx in x if id_to_token[idx.item()] != "<PAD>"]
    # print(x)
    test_documents.append(" ".join(x))

# test_documents
len(test_documents), len((result > 0.0).int().tolist()), len(y.tolist())
result_df = pd.DataFrame({
    "document": test_documents,
    "result": (result > 0.0).int().tolist(),
    "target": y.tolist()
})
result_df

,document,result,target
0,결말 이 아쉽 긴 했 다 . . 그래도 꼭 봐야 할 영화 ! !!!,1,1.0
1,평점 왤캐 낫 지 . . 단순 좀 비 영화 <UNK> 가 아닌 연기 와 스토리 거기...,1,1.0
2,이 정도 면 수작 이 지 . .,1,1.0
3,꼭 봐야 되 는 영화,1,1.0
4,왠지 김연아 크 면 에리카 처럼 될 것 같 음 .,1,1.0
5,멜로 임 ? 에로 임 ? ???,1,1.0
6,히치콕 이 이 영화 를 봤 다면 분명 박수 를 쳤 을 듯 . ..,1,1.0
7,간만에 심장 쫄리 고 결말 도 맘 에 듬 / 여자 가 무모 했 지만 ㅋ,1,1.0
8,홍상수 영화 랑 거의 비슷 해요 . 너무너무 재밌 게 봤 어요 .,1,1.0
9,어린 시절 내 배꼽 을 <UNK> 까지 날려 버린 영화,1,1.0


In [90]:
df[df["label"] == 0]

,id,document,label


In [80]:
tmp = []
tmp += " ".join(["a", "b", "c"])
tmp

['a', ' ', 'b', ' ', 'c']